# 02 — Data Cleaning

**Objective:** Clean the raw dataset by handling missing values, duplicates, data type conversions, and feature engineering.

**Output:** Cleaned dataset saved to `data/processed/cleaned_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = os.path.join('..', 'data', 'raw', 'data.csv')
df = pd.read_csv(RAW_PATH, encoding='ISO-8859-1')
print(f'Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2.1 Drop Missing CustomerID

CustomerID is essential for customer-level analytics. Rows without it cannot be used for segmentation or retention analysis.

In [ ]:
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f'Dropped {before - len(df):,} rows with missing CustomerID')
print(f'Remaining: {len(df):,} rows')

## 2.2 Remove Negative Quantity

Negative quantities represent cancellations/returns (InvoiceNo starts with 'C'). We remove these for clean revenue analysis.

In [ ]:
print(f'Rows with Quantity <= 0: {(df["Quantity"] <= 0).sum():,}')
df = df[df['Quantity'] > 0]
print(f'After removal: {len(df):,} rows')

## 2.3 Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df):,} duplicate rows')
print(f'Remaining: {len(df):,} rows')

## 2.4 Convert InvoiceDate to Datetime

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'InvoiceDate dtype: {df["InvoiceDate"].dtype}')
print(f'Date range: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')

## 2.5 Convert CustomerID to Integer

In [ ]:
df['CustomerID'] = df['CustomerID'].astype(int)
print(f'CustomerID dtype: {df["CustomerID"].dtype}')
print(f'Unique customers: {df["CustomerID"].nunique():,}')

## 2.6 Create Revenue Column

`Revenue = Quantity × UnitPrice`

In [ ]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print(f'Total Revenue: £{df["Revenue"].sum():,.2f}')
print(f'Revenue column stats:')
df['Revenue'].describe()

## 2.7 Add Time Features

In [ ]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
print('Added: Year, Month, DayOfWeek, Hour')
df.head()

## 2.8 Final Cleaned Dataset Summary

In [ ]:
print(f'Final shape: {df.shape}')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())

## 2.9 Save Cleaned Data

In [ ]:
output_path = os.path.join('..', 'data', 'processed', 'cleaned_data.csv')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f'Cleaned data saved to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1e6:.1f} MB')

## Cleaning Summary

| Step | Action | Rows Removed |
|------|--------|--------------|
| 1 | Drop missing CustomerID | ~135,080 |
| 2 | Remove negative/zero Quantity | ~8,905 |
| 3 | Remove duplicates | ~5,192 |
| 4 | Convert InvoiceDate | — |
| 5 | Create Revenue column | — |
| 6 | Add time features | — |

**Final dataset: 392,732 rows × 13 columns**